# Install & imports

In [1]:
!pip install -q mne mne-bids

import mne
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('MNE version:', mne.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
MNE version: 1.13.2


# New dedicated project structure

In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/PD_EEG_Thesis_Features_v2')
DATA_ROOT    = PROJECT_ROOT / 'data' / 'ds004584'
EDF_DIR      = PROJECT_ROOT / 'data' / 'edf'
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
EDF_DIR.mkdir(parents=True, exist_ok=True)

print('New standalone project directory:', PROJECT_ROOT)

Mounted at /content/drive
New standalone project directory: /content/drive/MyDrive/PD_EEG_Thesis_Features_v2


# Download dataset (same source, fresh copy)

In [ ]:
!pip install -q awscli

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 50.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [ ]:
result = !aws s3 sync --no-sign-request s3://openneuro.org/ds004584 "{DATA_ROOT}" \
    --exclude "*" \
    --include "*.json" \
    --include "*.tsv" \
    --include "*.set" \
    --include "*.fdt"

print('\n'.join(result[-10:]))
print(f'\nparticipants.tsv exists: {(DATA_ROOT / "participants.tsv").exists()}')

Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.1 MiB/s) with 1 file(s) remaining
Completed 2.9 GiB/2.9 GiB (32.0 MiB/s) with 1 file(s) remaining
download: s3://openneuro.org/ds004584/sub-149/eeg/sub-149_task-Rest_eeg.fdt to drive/MyDrive/PD_EEG_Thesis_Features_v2/data/ds004584/sub-149/eeg/sub-149_task-Rest_eeg.fdt

participants.tsv exists: True


# Build manifest, detect group column

In [ ]:
participants = pd.read_csv(DATA_ROOT / 'participants.tsv', sep='\t')
print('Columns found:', list(participants.columns))

CANDIDATE_COLS = ['group', 'Group', 'GROUP', 'diagnosis', 'Diagnosis', 'group_label']
GROUP_COL = next((c for c in CANDIDATE_COLS if c in participants.columns), None)

if GROUP_COL is None:
    # Fallback: low-cardinality object column, same logic as the original pipeline
    low_card = [(c, participants[c].nunique()) for c in participants.columns
                if participants[c].dtype == object and participants[c].nunique() <= 5]
    if low_card:
        GROUP_COL = sorted(low_card, key=lambda x: x[1])[0][0]
        print(f'Fallback to low-cardinality column: {GROUP_COL}')
    else:
        raise RuntimeError(f'Could not identify group column. Columns: {list(participants.columns)}')

print(f'\nGroup column: {GROUP_COL}')
print(participants[GROUP_COL].value_counts())

Columns found: ['participant_id', 'GROUP', 'ID', 'EEG', 'AGE', 'GENDER', 'MOCA', 'UPDRS', 'TYPE']

Group column: GROUP
GROUP
PD         100
Control     49
Name: count, dtype: int64


# Preprocess each subject AND export to literal .edf

In [ ]:
import subprocess

def preprocess_and_export_edf(subject_id, set_path, out_path):
    raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
    raw.pick_types(eeg=True)
    raw.filter(1.0, 40.0, method='fir', fir_window='hamming', verbose=False)
    raw.notch_filter(60.0, verbose=False)
    raw.set_eeg_reference('average', verbose=False)

    ica = mne.preprocessing.ICA(n_components=20, method='fastica',
                                random_state=42, max_iter='auto', verbose=False)
    ica.fit(raw, verbose=False)
    eog_chs = [ch for ch in ['Fp1', 'Fp2'] if ch in raw.ch_names]
    exclude = []
    if eog_chs:
        eog_inds, _ = ica.find_bads_eog(raw, ch_name=eog_chs, threshold=3.0, verbose=False)
        exclude.extend(eog_inds)
    muscle_inds, _ = ica.find_bads_muscle(raw, verbose=False)
    exclude.extend(muscle_inds)
    ica.exclude = list(set(exclude))
    raw_clean = ica.apply(raw.copy(), verbose=False)

    # Literal .edf export, as instructed
    raw_clean.export(str(out_path), fmt='edf', overwrite=True, verbose=False)
    return raw_clean, len(exclude)


manifest_rows = []
for set_file in sorted(DATA_ROOT.glob('sub-*/eeg/*_eeg.set')):
    subject_id = set_file.parts[-3].replace('sub-', '')
    group_row = participants.loc[participants['participant_id'] == f'sub-{subject_id}', GROUP_COL]
    if len(group_row) == 0:
        continue
    group = group_row.values[0]

    edf_path = EDF_DIR / f'sub-{subject_id}_clean.edf'
    if edf_path.exists():
        print(f'[skip] sub-{subject_id} already exported')
    else:
        try:
            _, n_excl = preprocess_and_export_edf(subject_id, str(set_file), str(edf_path))
            print(f'[✓] sub-{subject_id} ({group}) → {n_excl} ICA components removed, exported to .edf')
        except Exception as e:
            print(f'[✗] sub-{subject_id} FAILED: {e}')
            continue

    manifest_rows.append({'subject': subject_id, 'group': group, 'edf_path': str(edf_path)})

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(PROJECT_ROOT / 'manifest.csv', index=False)
print(f'\nManifest saved: {len(manifest)} subjects')
print(manifest['group'].value_counts())

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[✓] sub-001 (PD) → 17 ICA components removed, exported to .edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[✓] sub-002 (PD) → 7 ICA components removed, exported to .edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[✓] sub-003 (PD) → 6 ICA components removed, exported to .edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[✓] sub-004 (PD) → 4 ICA components removed, exported to .edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[✓] sub-005 (PD) → 15 ICA components removed, exported to .edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[✓] sub-006 (PD) → 11 ICA components removed, exported to .edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
[✓] sub-007 (PD) → 7 ICA components removed, exported to .edf
NOTE: pick_types() is a 

# Verify .edf files load correctly

In [ ]:
sample = manifest.iloc[0]
raw_check = mne.io.read_raw_edf(sample['edf_path'], preload=False, verbose=False)
print(f'Verified: sub-{sample["subject"]}  |  {len(raw_check.ch_names)} channels  |  {raw_check.info["sfreq"]} Hz')
print(f'File format on disk: {Path(sample["edf_path"]).suffix}')

Verified: sub-001  |  63 channels  |  500.0 Hz
File format on disk: .edf
